# Drosophila Brain Cocaine Response - Notebook 3: Differential Expression & Enrichment

Resumes from the Notebook 2 checkpoint (`checkpoint_02_clustered.h5ad`) - clustered AnnData with Leiden assignments and cluster markers computed.

## Setup: reload paths and checkpoint from Notebook 2

In [ ]:
from pathlib import Path
import sys, gc, math

import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(r"D:\bmp\sc_project2_drosophila_brain")
DATA_DIR = PROJECT_ROOT / "data"
RESULT_DIR = PROJECT_ROOT / "results"
FIG_DIR = RESULT_DIR / "figures"
TABLES_DIR = RESULT_DIR / "tables"

RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=120, facecolor="white", frameon=False)
sc.settings.figdir = str(FIG_DIR)

checkpoint_path = RESULT_DIR / "checkpoint_02_clustered.h5ad"
adata = sc.read_h5ad(checkpoint_path)
print(f"Loaded checkpoint: {adata.n_obs} cells x {adata.n_vars} genes, "
      f"{adata.obs['leiden_res_0.8'].nunique()} clusters")

## Step 11: Differential expression — Cocaine vs. Sucrose, by sex
Publication threshold: |log2FC| > 1.0 and adjusted p < 0.05. Keeps both the full (`_de`) and significant-only (`_sig`) tables — the full tables are needed for the volcano plots in Step 13.

In [ ]:
def sex_specific_de(adata_all, sex_name):
    subset = adata_all[adata_all.obs["sex"].astype(str) == sex_name].copy()
    sc.tl.rank_genes_groups(subset, groupby="treatment", reference="Sucrose", method="wilcoxon")
    de = sc.get.rank_genes_groups_df(subset, group="Cocaine")
    de["significant"] = de["logfoldchanges"].abs().gt(1.0) & de["pvals_adj"].lt(0.05)
    return subset, de

adata_male, male_de = sex_specific_de(adata, "Male")
adata_female, female_de = sex_specific_de(adata, "Female")

male_sig = male_de[male_de["significant"]].copy()
female_sig = female_de[female_de["significant"]].copy()

print("Male significant genes:", len(male_sig))
print("Female significant genes:", len(female_sig))

male_de.to_csv(TABLES_DIR / "male_Cocaine_vs_Sucrose_all_DE.csv", index=False)
female_de.to_csv(TABLES_DIR / "female_Cocaine_vs_Sucrose_all_DE.csv", index=False)
male_sig.to_csv(TABLES_DIR / "male_Cocaine_vs_Sucrose_significant_DE.csv", index=False)
female_sig.to_csv(TABLES_DIR / "female_Cocaine_vs_Sucrose_significant_DE.csv", index=False)

## Step 12: Male/female DE overlap (sexual dimorphism check)

In [ ]:
male_genes = set(male_sig["names"].astype(str))
female_genes = set(female_sig["names"].astype(str))
shared_genes = sorted(male_genes & female_genes)
male_only = sorted(male_genes - female_genes)
female_only = sorted(female_genes - male_genes)

overlap = pd.DataFrame({
    "category": ["Male-only", "Female-only", "Shared"],
    "n_genes": [len(male_only), len(female_only), len(shared_genes)],
})
display(overlap)

pd.DataFrame({"gene": shared_genes}).to_csv(TABLES_DIR / "shared_DE_genes.csv", index=False)
pd.DataFrame({"gene": male_only}).to_csv(TABLES_DIR / "male_only_DE_genes.csv", index=False)
pd.DataFrame({"gene": female_only}).to_csv(TABLES_DIR / "female_only_DE_genes.csv", index=False)

## Step 13: Volcano plots (Figure 4)

In [ ]:
def volcano(de, title, filename, label_top_n=5):
    df = de.copy()
    df["neglog10_padj"] = -np.log10(df["pvals_adj"].clip(lower=np.finfo(float).tiny))
    sig = df["logfoldchanges"].abs().gt(1) & df["pvals_adj"].lt(0.05)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(df.loc[~sig, "logfoldchanges"], df.loc[~sig, "neglog10_padj"], s=8, alpha=0.35, label="Not significant")
    ax.scatter(df.loc[sig, "logfoldchanges"], df.loc[sig, "neglog10_padj"], s=10, alpha=0.65, color="crimson", label="Significant")
    ax.axvline(1, ls="--", lw=1, color="gray")
    ax.axvline(-1, ls="--", lw=1, color="gray")
    ax.axhline(-np.log10(0.05), ls="--", lw=1, color="gray")

    label_df = df.loc[sig].reindex(df.loc[sig, "logfoldchanges"].abs().sort_values(ascending=False).index)
    label_df = label_df.head(label_top_n).sort_values("neglog10_padj", ascending=False).reset_index(drop=True)

    golden_angle = 137.508
    base_radius = 18
    for i, row in label_df.iterrows():
        angle_rad = math.radians((i * golden_angle) % 360)
        radius = base_radius + 14 * i
        dx, dy = radius * math.cos(angle_rad), radius * math.sin(angle_rad)
        ax.annotate(
            str(row["names"]),
            xy=(row["logfoldchanges"], row["neglog10_padj"]),
            xytext=(dx, dy),
            textcoords="offset points",
            fontsize=8,
            ha="center",
            arrowprops=dict(arrowstyle="-", lw=0.6, color="gray", shrinkA=2, shrinkB=2),
        )

    ax.set_xlabel("log2 fold change")
    ax.set_ylabel("-log10 adjusted p-value")
    ax.set_title(title)
    ax.legend(loc="upper right", fontsize=8)
    plt.tight_layout()

    output = FIG_DIR / filename
    plt.savefig(output, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", output)

volcano(male_de, "Male: Cocaine vs Sucrose", "male_volcano.png")
volcano(female_de, "Female: Cocaine vs Sucrose", "female_volcano.png")

## Step 14: Pathway enrichment with GSEApy (correct Fly-Enrichr library names)

In [ ]:
import gseapy as gp
print("GSEApy:", gp.__version__)

fly_libraries = gp.get_library_name(organism="Fly")
go_bp = [x for x in fly_libraries if "GO_Biological_Process" in x and "20" in x]
kegg = [x for x in fly_libraries if "KEGG" in x]
gene_sets = (go_bp[-1:] if go_bp else []) + (kegg[-1:] if kegg else [])
print("Using gene sets:", gene_sets)

def enrich(sig_df, label):
    genes = sig_df["names"].astype(str).drop_duplicates().tolist()[:150]
    if not genes:
        print(f"No significant genes for {label}, skipping enrichment.")
        return None
    try:
        result = gp.enrichr(gene_list=genes, gene_sets=gene_sets, organism="fly", outdir=None)
        result.results.to_csv(TABLES_DIR / f"{label.lower()}_enrichment_results.csv", index=False)
        return result
    except Exception as e:
        print(f"Enrichment failed for {label}: {e}")
        return None

enr_male = enrich(male_sig, "Male")
enr_female = enrich(female_sig, "Female")

if enr_male is not None:
    print("Top enriched pathways in Males:")
    display(enr_male.results.head(10)[["Gene_set", "Term", "Adjusted P-value", "Genes"]])
if enr_female is not None:
    print("Top enriched pathways in Females:")
    display(enr_female.results.head(10)[["Gene_set", "Term", "Adjusted P-value", "Genes"]])

## Step 15: Pathway enrichment bar charts (Figure 5)

In [ ]:
for sex in ["male", "female"]:
    file = TABLES_DIR / f"{sex}_enrichment_results.csv"
    if not file.exists():
        print(f"No enrichment results file for {sex}, skipping.")
        continue

    df = pd.read_csv(file)
    df = df.dropna(subset=["Term", "Adjusted P-value"])
    df = df[df["Adjusted P-value"] > 0].sort_values("Adjusted P-value").head(10)
    df["neglog10_padj"] = -np.log10(df["Adjusted P-value"])

    n_significant = int((df["Adjusted P-value"] < 0.05).sum())
    print(f"{sex.capitalize()}: {n_significant} of top {len(df)} shown terms pass adjusted p < 0.05")

    plt.figure(figsize=(8, 5))
    plt.barh(df["Term"].iloc[::-1], df["neglog10_padj"].iloc[::-1])
    plt.axvline(-np.log10(0.05), color="red", ls="--", lw=1, label="adj p = 0.05")
    plt.xlabel("-log10 adjusted p-value")
    plt.title(f"{sex.capitalize()} pathway enrichment")
    plt.legend()
    plt.tight_layout()

    output = FIG_DIR / f"{sex}_pathway_enrichment.png"
    plt.savefig(output, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", output)

## Step 16: Final summary and export

In [ ]:
summary = pd.DataFrame({
    "metric": ["cells_after_QC_subsampled", "genes_after_QC", "HVGs", "Leiden_clusters",
               "male_significant_DE_genes", "female_significant_DE_genes", "shared_DE_genes"],
    "value": [adata.n_obs, adata.n_vars, int(adata.var["highly_variable"].sum()) if "highly_variable" in adata.var else adata.n_vars,
              adata.obs["leiden_res_0.8"].nunique(), len(male_sig), len(female_sig), len(shared_genes)],
})
display(summary)
summary.to_csv(RESULT_DIR / "final_analysis_summary.csv", index=False)
adata.write_h5ad(RESULT_DIR / "drosophila_cocaine_processed.h5ad")

print("Final AnnData:", RESULT_DIR / "drosophila_cocaine_processed.h5ad")
print("Figures:", FIG_DIR)
print("Results:", RESULT_DIR)
print()
print("REMEMBER for your report: this run used a stratified subsample (~", adata.n_obs,
      "cells) of the 88,923 QC-passed cells due to local memory constraints.")